# Question 1: Employment Statistics Analysis

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Question:** Using the NYS DOL's employment statistics data for the latest available month:
- **1a)** Which major industries (by 2-digit NAICS) in New York City changed the most over the prior year? Describe possible reasons why.
- **1b)** How was the 1-year change different from the 5-year change?

**Data Source:** NYS DOL Current Employment Statistics (`ces.csv`), not seasonally adjusted.
**Latest Available Month:** February 2026.
**Geography:** New York City (5 boroughs).

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ces = pd.read_csv('../data/ces.csv')
ces.columns = ces.columns.str.strip()
nyc = ces[ces['AREANAME'] == 'New York City'].copy()

---
## Step 2.1: Data Preparation

Pull the 10 BLS-defined supersectors and Total Nonfarm directly from the NYS DOL CES data.

**Data notes:**
- ces.csv is **not seasonally adjusted** — we only compare same-month across years
- Employment figures are in **actual units** (not thousands)
- All 10 supersectors are official CES series — no values are computed or hardcoded
- The 10 supersectors sum exactly to Total Nonfarm with no overlap


In [ ]:
supersectors = [
    ('21+23', 'Mining, Logging and Construction'),
    ('31-33', 'Manufacturing'),
    ('22,42-49', 'Trade, Transportation, and Utilities'),
    ('51', 'Information'),
    ('52,53', 'Financial Activities'),
    ('54-56', 'Professional and Business Services'),
    ('61,62', 'Private Education and Health Services'),
    ('71,72', 'Leisure and Hospitality'),
    ('81', 'Other Services'),
    ('92', 'Government'),
]

short = {
    'Mining, Logging and Construction': 'Mining/Construction',
    'Manufacturing': 'Manufacturing',
    'Trade, Transportation, and Utilities': 'Trade/Trans/Util',
    'Information': 'Information',
    'Financial Activities': 'Financial Activities',
    'Professional and Business Services': 'Prof & Business',
    'Private Education and Health Services': 'Education & Health',
    'Leisure and Hospitality': 'Leisure & Hospitality',
    'Other Services': 'Other Services',
    'Government': 'Government',
}

rows = []
for naics, title in supersectors:
    feb26 = nyc[(nyc['YEAR'] == 2026) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
    feb25 = nyc[(nyc['YEAR'] == 2025) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
    feb21 = nyc[(nyc['YEAR'] == 2021) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
    if len(feb26) > 0 and len(feb25) > 0 and len(feb21) > 0:
        v26, v25, v21 = int(feb26.values[0]), int(feb25.values[0]), int(feb21.values[0])
        rows.append({
            'NAICS': naics, 'Industry': title,
            'Feb 2026': v26, 'Feb 2025': v25, 'Feb 2021': v21,
            'YoY Change': v26 - v25, 'YoY %': round((v26 - v25) / v25 * 100, 2),
            '5yr Change': v26 - v21, '5yr %': round((v26 - v21) / v21 * 100, 2),
        })

tnf26 = int(nyc[(nyc['YEAR'] == 2026) & (nyc['INDUSTRY_TITLE'] == 'Total Nonfarm')]['FEB'].values[0])
tnf25 = int(nyc[(nyc['YEAR'] == 2025) & (nyc['INDUSTRY_TITLE'] == 'Total Nonfarm')]['FEB'].values[0])
tnf21 = int(nyc[(nyc['YEAR'] == 2021) & (nyc['INDUSTRY_TITLE'] == 'Total Nonfarm')]['FEB'].values[0])
rows.append({
    'NAICS': '', 'Industry': 'TOTAL NONFARM',
    'Feb 2026': tnf26, 'Feb 2025': tnf25, 'Feb 2021': tnf21,
    'YoY Change': tnf26 - tnf25, 'YoY %': round((tnf26 - tnf25) / tnf25 * 100, 2),
    '5yr Change': tnf26 - tnf21, '5yr %': round((tnf26 - tnf21) / tnf21 * 100, 2),
})

df = pd.DataFrame(rows)

def color_change(val):
    if isinstance(val, (int, float)):
        return 'color: green; font-weight: bold' if val > 0 else ('color: red; font-weight: bold' if val < 0 else '')
    return ''

print(f'Supersectors from CES: {len(df) - 1} + Total Nonfarm')
print(f'Sum check: {df.iloc[:-1]["Feb 2026"].sum():,} vs Total Nonfarm {tnf26:,} -> {"MATCH" if df.iloc[:-1]["Feb 2026"].sum() == tnf26 else "MISMATCH"}')


---
## Step 2.2: Year-over-Year Changes (Q1a)

Rank industries by percentage change. All values pulled directly from NYS DOL CES data.

### BLS Supersector Groupings

The sectors follow the **BLS NAICS supersector classification** defined at:
https://www.bls.gov/sae/additional-resources/naics-supersectors-for-ces-program.htm

Five supersectors are aggregations of sub-sectors (also available in CES):
- **Trade/Trans/Util** (BLS code 40): NAICS 22 + 42 + 44-45 + 48-49
- **Financial Activities** (BLS code 55): NAICS 52 + 53
- **Prof & Business** (BLS code 60): NAICS 54 + 55 + 56
- **Education & Health** (BLS code 65): NAICS 61 + 62
- **Leisure & Hospitality** (BLS code 70): NAICS 71 + 72


In [ ]:
yoy_cols = ['NAICS', 'Industry', 'Feb 2026', 'Feb 2025', 'YoY Change', 'YoY %']
df_yoy = df[yoy_cols].sort_values('YoY %', ascending=False).reset_index(drop=True)
df_yoy['Industry'] = df_yoy['Industry'].map(short)

display(df_yoy.style.format({
    'Feb 2026': '{:,}', 'Feb 2025': '{:,}', 'YoY Change': '{:+,}', 'YoY %': '{:+.2f}%'
}).map(color_change, subset=['YoY Change', 'YoY %']).hide(axis='index'))

priv26 = tnf26 - df.loc[df['NAICS'] == '92', 'Feb 2026'].values[0]
priv25 = tnf25 - df.loc[df['NAICS'] == '92', 'Feb 2025'].values[0]
print(f'Private sector: Feb 2026 = {priv26:,} | YoY change = {priv26 - priv25:+,}')


---
## Step 2.3: Five-Year Comparison Table (Q1b)

**CRITICAL CONTEXT:** Feb 2021 was still in COVID disruption. Large positive 5-year % changes reflect recovery from COVID lows, not sustainable growth rates.


In [ ]:
five_yr_cols = ['NAICS', 'Industry', 'Feb 2026', 'Feb 2021', '5yr Change', '5yr %']
df_5yr = df[five_yr_cols].sort_values('5yr %', ascending=False).reset_index(drop=True)
df_5yr['Industry'] = df_5yr['Industry'].map(short)

display(df_5yr.style.format({
    'Feb 2026': '{:,}', 'Feb 2021': '{:,}',
    '5yr Change': '{:+,}', '5yr %': '{:+.2f}%'
}).map(color_change, subset=['5yr Change', '5yr %']).hide(axis='index'))


---
## Step 2.4: Chart 1 — YoY % Change by Industry

All 10 BLS supersectors plus Total Nonfarm.


In [ ]:
chart_df = df.sort_values('YoY %', ascending=True).copy()
chart_df['label'] = chart_df['NAICS'] + ': ' + chart_df['Industry'].map(short)

colors = ['#d32f2f' if v < 0 else '#2e7d32' for v in chart_df['YoY %']]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=chart_df['label'], x=chart_df['YoY %'], orientation='h',
    marker_color=colors,
    text=[f'{v:+.2f}%' for v in chart_df['YoY %']],
    textposition='outside', textfont_size=10, cliponaxis=False,
))

fig.update_layout(
    title='NYC 1-Year Employment % Change by Industry (Feb 2025 -> Feb 2026)<br><sup>Source: NYS DOL CES | All values from BLS supersector series</sup>',
    xaxis_title='YoY % Change', yaxis_title='',
    height=500, margin=dict(l=200, r=60, t=80, b=50), font=dict(size=11),
)
fig.show()


---
## Step 2.5: Chart 2 — 1-Year vs 5-Year % Change Comparison

Highlights the gap between COVID recovery (5yr) and current trend (1yr).


In [ ]:
comp_df = df[df['NAICS'] != ''].copy()
comp_df['label'] = comp_df['NAICS'] + ': ' + comp_df['Industry'].map(short)
comp_df = comp_df.sort_values('5yr %')

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=comp_df['label'], x=comp_df['YoY %'], name='1-Year',
    orientation='h', marker_color='#1565c0',
))
fig2.add_trace(go.Bar(
    y=comp_df['label'], x=comp_df['5yr %'], name='5-Year',
    orientation='h', marker_color='#f57c00',
))

fig2.update_layout(
    barmode='group', height=600,
    title='1-Year vs 5-Year Employment % Change by Sector<br><sup>Sorted by 5-year change | COVID recovery distortion visible in large gaps</sup>',
    xaxis_title='% Change', yaxis_title='',
    margin=dict(l=220, r=60, t=80, b=50),
    legend=dict(x=1, y=0, xanchor='right', yanchor='bottom'),
    font=dict(size=10),
)
fig2.show()


---
## Step 2.6: Chart 3 — 5-Year Employment Trends (2021–2026)

Interactive trend lines for each supersector. Click legend items to toggle.


In [ ]:
import plotly.colors as pc

years_recent = sorted([y for y in nyc['YEAR'].unique() if 2021 <= y <= 2026])

def get_feb(title, years):
    vals = []
    for yr in years:
        v = nyc[(nyc['YEAR'] == yr) & (nyc['INDUSTRY_TITLE'] == title)]['FEB']
        vals.append(int(v.values[0]) if len(v) > 0 and pd.notna(v.values[0]) else None)
    return vals

palette = pc.qualitative.Set2

fig3 = go.Figure()
for i, (_, row) in enumerate(df[df['NAICS'] != ''].iterrows()):
    title = row['Industry']
    label = short.get(title, title)
    vals = get_feb(title, years_recent)
    fig3.add_trace(go.Scatter(
        x=years_recent, y=vals, name=label, mode='lines+markers',
        line=dict(color=palette[i % len(palette)], width=2),
    ))

tnf_vals = get_feb('Total Nonfarm', years_recent)
fig3.add_trace(go.Scatter(
    x=years_recent, y=tnf_vals, name='Total Nonfarm', mode='lines+markers',
    line=dict(color='black', width=3, dash='dash'),
    visible='legendonly',
))

fig3.update_layout(
    title='NYC Employment Trends by BLS Supersector (Feb 2021 - Feb 2026)<br><sup>Source: NYS DOL CES | Click legend to toggle sectors</sup>',
    xaxis_title='Year', yaxis_title='Employment',
    height=600, font=dict(size=11),
    legend=dict(font=dict(size=9)),
)
fig3.show()


---
## Step 2.7: Narrative — Possible Reasons for Industry Changes



<h2 style='text-decoration:underline'>1-Year Changes (Feb 2026 vs Feb 2025)</h2>

<h3 style='color:red'>Decliners</h3>

**Manufacturing (NAICS 31-33): <span style='color:red'>-5.82% (-3,100 jobs)</span>**

Manufacturing has been in long-term structural decline in NYC, driven by automation, offshoring of production, and the city's transition to a service-oriented economy.

**Mining/Construction (NAICS 21+23): <span style='color:red'>-4.82% (-6,600 jobs)</span>**

*BLS Supersector: Mining, Logging & Construction. CES provides this as a single combined series for NYC.*

Construction declined likely due to higher interest rates throughout 2025 that suppressed new housing starts and commercial development, along with the completion of several major infrastructure projects.

**Other Services (NAICS 81): <span style='color:red'>-2.77% (-4,900 jobs)</span>**

Includes repair services, personal care, and civic organizations. Inflation may be reducing consumer discretionary spending on non-essential services.

**Leisure & Hospitality (NAICS 71,72): <span style='color:red'>-2.28% (-10,000 jobs)</span>**

*Supersector composition: Arts, Entertainment & Recreation (NAICS 71) + Accommodation & Food Services (NAICS 72). BLS Supersector 70.*

The post-pandemic recovery in tourism, dining, and entertainment may have peaked, with consumers reallocating spending toward essentials.

**Education & Health (NAICS 61,62): <span style='color:red'>-1.62% (-21,400 jobs)</span>**

*Supersector composition: Private Educational Services (NAICS 61) + Health Care & Social Assistance (NAICS 62). BLS Supersector 65.*

The largest absolute YoY decline of any supersector. This reversal may reflect post-pandemic normalization as healthcare demand subsides.

**Trade/Trans/Util (NAICS 22,42-49): <span style='color:red'>-1.22% (-7,100 jobs)</span>**

*Supersector composition: Utilities (NAICS 22) + Wholesale Trade (42) + Retail Trade (44-45) + Transportation & Warehousing (48-49). BLS Supersector 40.*

<h3 style='color:green'>Growers</h3>

**Information (NAICS 51): <span style='color:green'>+1.93% (+4,200 jobs)</span>**

Driven by media streaming, digital content, and AI-related services. The NYC Comptroller (Feb 2026) projects benchmark revisions may revise some gains downward.

**Government (NAICS 92): <span style='color:green'>+1.58% (+9,500 jobs)</span>**

Driven by local government hiring, particularly in education, as the city fills positions left vacant during pandemic-era budget constraints. This is the only sector where 1-year and 5-year growth rates are similar, indicating steady rather than recovery-driven expansion.

**Financial Activities (NAICS 52,53): <span style='color:green'>+1.27% (+6,500 jobs)</span>**

*Supersector composition: Finance & Insurance (NAICS 52) + Real Estate & Rental/Leasing (NAICS 53). BLS Supersector 55.*

Steady expansion reflecting NYC's role as a global financial center.

**Prof & Business (NAICS 54-56): <span style='color:green'>+0.82% (+6,500 jobs)</span>**

*Supersector composition: Professional/Scientific/Technical (54) + Management of Companies (55) + Admin/Waste Management (56). BLS Supersector 60.*

**1-Year Takeaway:** NYC Total Nonfarm fell by -39,400 jobs (-0.82%) YoY to 4,791,000. Only 4 of 10 supersectors grew over the past year.

---
<h2 style='text-decoration:underline'>5-Year Changes (Feb 2026 vs Feb 2021)</h2>

**CRITICAL CONTEXT: Feb 2021 was still in COVID disruption.** The 5-year figures below are dominated by recovery from pandemic-era lows.

<h3 style='color:green'>Growers</h3>

**Leisure & Hospitality (NAICS 71,72): <span style='color:green'>+85.05% (+197,400 jobs)</span>**

*Supersector composition: Arts, Entertainment & Recreation (NAICS 71) + Accommodation & Food Services (NAICS 72). BLS Supersector 70.*

The aggregate supersector tells the same story — recovery from a Feb 2021 trough of just 232,100 jobs.

**Education & Health (NAICS 61,62): <span style='color:green'>+25.07% (+260,100 jobs)</span>**

*Supersector composition: Private Educational Services (NAICS 61) + Health Care & Social Assistance (NAICS 62). BLS Supersector 65.*

The largest absolute 5-year gain of any supersector. While partially COVID recovery, this sector also benefited from structural demand growth in healthcare and aging population needs.

**Prof & Business (NAICS 54-56): <span style='color:green'>+12.75% (+89,200 jobs)</span>**

*Supersector composition: Professional/Scientific/Technical (54) + Management of Companies (55) + Admin/Waste Management (56). BLS Supersector 60.*

**Financial Activities (NAICS 52,53): <span style='color:green'>+12.13% (+55,900 jobs)</span>**

*Supersector composition: Finance & Insurance (NAICS 52) + Real Estate & Rental/Leasing (NAICS 53). BLS Supersector 55.*

<h3 style='color:red'>Decliners</h3>

**Mining/Construction (NAICS 21+23): <span style='color:red'>-3.91% (-5,300 jobs)</span>**

*BLS Supersector: Mining, Logging & Construction.*

Despite the YoY narrative about interest rates, the 5-year picture is also negative — construction employment in Feb 2021 was still relatively elevated compared to other sectors, leaving less recovery upside.

**Manufacturing (NAICS 31-33): <span style='color:red'>-3.65% (-1,900 jobs)</span>**

The only sector besides Mining/Construction to decline over 5 years. Structural decline has continued unabated through the COVID cycle — manufacturing never benefited from recovery momentum.

**5-Year Takeaway:** The divergence is the story. Sectors that lost the most during COVID show the largest 5-year gains (recovery) but are now declining again (maturation). Sectors that were less disrupted show modest 5-year gains and steadier current performance.
